# Centralized Model (Baseline)

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from train import task
from exploration import exploration_utils
import centralized_utils

## Data

In [1]:
import pandas as pd
import glob
import os

In [5]:
hospitals = ['hospital_a', 'hospital_b', 'hospital_c']
hospital_data = {}

patterns = {
    'train': 'train-*.parquet',
    'eval':  'eval-*.parquet',
    'test':  'test*.parquet',
}

for hospital in hospitals:
    data_path = os.path.join('dataset/', hospital)
    hospital_data[hospital] = {}
    
    for split, pattern in patterns.items():
        files = glob.glob(os.path.join(data_path, pattern))
        hospital_data[hospital][split] = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

hospital_data['hospital_d'] = {}
hospital_d_files = glob.glob(os.path.join('../../data/', 'test*.parquet'))
hospital_data['hospital_d']['test'] = pd.concat([pd.read_parquet(f) for f in hospital_d_files], ignore_index=True)

ValueError: No objects to concatenate

In [ ]:
rows = {}

for split in patterns.keys():
    rows[split] = {
        'Hospital A': len(hospital_data['hospital_a'][split]),
        'Hospital B': len(hospital_data['hospital_b'][split]), 
        'Hospital C': len(hospital_data['hospital_c'][split]),
        'Hospital D': len(hospital_data['hospital_d'][split]) if split == 'test' else 0,
    }
    rows[split]['Total'] = sum(rows[split][h] for h in ['Hospital A', 'Hospital B', 'Hospital C', 'Hospital D'])
rows['Total'] = {
    'Hospital A': sum(rows[split]['Hospital A'] for split in ['train', 'eval', 'test']),
    'Hospital B': sum(rows[split]['Hospital B'] for split in ['train', 'eval', 'test']), 
    'Hospital C': sum(rows[split]['Hospital C'] for split in ['train', 'eval', 'test']),
    'Hospital D': sum(rows[split]['Hospital D'] for split in ['train', 'eval', 'test'])
}

rows['Total']['Total'] = sum(rows['Total'][hospital] for hospital in ['Hospital A', 'Hospital B', 'Hospital C', 'Hospital D'])

data_info = pd.DataFrame.from_dict(rows, orient='index')
data_info

In [ ]:
df_train_combined = pd.concat([hospital_data[h]['train'] for h in hospital_data if h != 'hospital_d'], ignore_index=True)
df_val_combined = pd.concat([hospital_data[h]['eval'] for h in hospital_data if h != 'hospital_d'], ignore_index=True)
df_test_combined = pd.concat([hospital_data[h]['test'] for h in hospital_data], ignore_index=True)
len(df_train_combined), len(df_val_combined), len(df_test_combined)

### Class Imbalance

In [ ]:
label_distribution = df_train_combined['label'].explode().value_counts().reset_index()
label_distribution.columns = ['Pathology Label', 'Frequency']
label_distribution['Percentage (%)'] = (label_distribution['Frequency'] / len(df_train_combined) * 100).map(lambda x: f'{x:.2f}')
label_distribution

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=label_distribution, x='Pathology Label', y='Frequency', hue='Pathology Label', ax=ax[0], palette='colorblind')
ax[0].set_title('Distribution of Thoracic Diseases').set_weight('bold')
ax[0].set_xlabel('Classes')
ax[0].set_ylabel('Number of Examples')
ax[0].tick_params(axis='x', rotation=40)

table = ax[1].table(cellText=label_distribution.values, cellLoc='center', 
            colLabels=label_distribution.columns, loc='center', colWidths=[0.5, 0.5, 0.3])
table.scale(0.9 ,2)
table[0, 0].get_text().set_weight('bold')
table[0, 1].get_text().set_weight('bold')
table[0, 2].get_text().set_weight('bold')
ax[1].axis('off')
plt.tight_layout()
plt.show();

### Data Visualization (Preprocessing vs. No-preprocessing)

In [ ]:
from torchvision import models

Desnet121 requires the following data preprocessing:

In [ ]:
weights = models.DenseNet121_Weights.DEFAULT
preprocess_transform = weights.transforms()
preprocess_transform

In [ ]:
train_preprocessed = task.XrayDataset(df_train_combined, transform=preprocess_transform)
train_not_preprocessed = task.XrayDataset(df_train_combined, transform=None)

In [ ]:
pre_img, pre_label = train_preprocessed[5]
not_img, not_label = train_not_preprocessed[5]

#### Preprocessed

In [ ]:
centralized_utils.show_xray(pre_img.permute(1, 2, 0), pre_label)

#### Not Preprocessed

In [ ]:
centralized_utils.show_xray(not_img, not_label)

### Weights

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import torch

In [ ]:
unique_labels = set(label for labels in df_train_combined['label'] for label in labels)
print(unique_labels)

In [ ]:
df_train_combined['label'] = df_train_combined['label'].apply(lambda x: [] if 'No Finding' in x else x)

In [ ]:
unique_labels = set(label for labels in df_train_combined['label'] for label in labels)
print(unique_labels) # No Finding not present

In [ ]:
mlb = MultiLabelBinarizer(classes=task.DISEASE_LABELS)
y_np = mlb.fit_transform(df_train_combined['label'])

In [ ]:
# Double Checking one-hot encoding is consistent with DISEASE_LABELS
np.array_equal(mlb.classes_, task.DISEASE_LABELS)

In [ ]:
y = torch.tensor(y_np, dtype=torch.float32)

In [ ]:
pos_weights, neg_weights = centralized_utils.calculate_weights(y)
pos_weights, neg_weights

In [ ]:
pos_weights = centralized_utils.calculate_pos_weights(y)
pos_weights

## Model

#### New Loss Function - Weighted Binary Cross Entropy Loss Function

In [ ]:
def weighted_bce_loss(p, y, pos_weights, neg_weights, epsilon=1e-7):
    L = -(pos_weights * y * torch.log(p + epsilon) + neg_weights * (1 - y) * torch.log(1 - p + epsilon))
    return L.mean()

In [ ]:
import torch
import time

In [ ]:
device = 'cpu'
if torch.cuda.is_available():
    device = torch.device('cuda:0')
print('Running on', device)

In [ ]:
n_classes = 14
densenet121 = centralized_utils.DenseNet121(n_classes, device=device)
densenet121

## Training

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
from train import task

from sklearn.metrics import jaccard_score, f1_score, recall_score, precision_score, roc_auc_score, roc_curve, average_precision_score, RocCurveDisplay
import matplotlib.pyplot as plt
from torchvision import models
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
import time
import glob
import os

In [ ]:
def train(model, train_loader, val_loader, device, epochs, pos_weight=None, learning_rate=0.001):
    """
    Training loop that outputs loss, training time, and peak memory usage.
    """
    if pos_weight is None:
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    total_train_loss = []
    total_val_loss = []
    
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    total_training_time = 0.0
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f'Train - Epoch {epoch+1}/{epochs}')

        train_start = time.time()
        for images, labels in train_pbar:
            images, labels = images.to(device), labels.to(device)
        
            logits = model(images)
            loss = criterion(logits, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        train_time = time.time() - train_start
        total_training_time += train_time
        avg_train_loss = train_loss / len(train_loader)
        total_train_loss.append(avg_train_loss)

        # Validation
        val_loss = 0.0
        model.eval()
        eval_pbar = tqdm(val_loader, desc=f'Val - Epoch {epoch+1}/{epochs}')
        with torch.no_grad():
            for images, labels in eval_pbar:
                images, labels = images.to(device), labels.to(device)
                
                logits = model(images)
                loss = criterion(logits, labels)
                    
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        total_val_loss.append(avg_val_loss)
        
        print(f'Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Train Time: {train_time:.2f}s')

    if torch.cuda.is_available():
        peak_memory_usage = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    else:
        peak_memory_usage = None
    
    return total_train_loss, total_val_loss, total_training_time, peak_memory_usage

In [ ]:
n_epochs = 15
batch_size = 32
learning_rate = 0.0001

n_classes = 14
densenet121 = centralized_utils.DenseNet121(n_classes, device=device)
pos_weights = pos_weights.to(device)

train_dataset = task.XrayDataset(df_train_combined, transform=preprocess_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = task.XrayDataset(df_val_combined, transform=preprocess_transform)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
total_train_loss, total_val_loss, training_time, peak_memory_usage = train(densenet121, train_loader, val_loader, 
                                                                           device, n_epochs, pos_weights, learning_rate);

## Evaluation

In [ ]:
def threshold_tuning(model, val_loader):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            logits = model(images)

            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    num_classes = all_labels.shape[1]
    best_thresholds = []
    thresholds = np.linspace(0, 1, 101)
    for label in range(num_classes):
        best_t = 0.5
        best_score = -1

        for t in thresholds:
            preds = (all_probs[:, label] >= t).astype(int) # T/F -> 0/1
            score = f1_score(all_labels[:, label], preds)

            if score > best_score:
                 best_score = score
                 best_t = t
        best_thresholds.append(best_t)
    return np.array(best_thresholds) 

### Training Loss Curve

In [ ]:
centralized_utils.plot_loss(total_train_loss)

### Metrics

In [ ]:
from sklearn.metrics import jaccard_score, f1_score, recall_score, precision_score, roc_auc_score, roc_curve, average_precision_score, classification_report
import matplotlib.pyplot as plt
from torchvision import models
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
import time
import glob
import os

In [ ]:
def evaluate(model, test_loader, device, thresholds):
    """
    Evaluates model using Jaccard, F1, Recall, Precision, ROC-AUC, Per-class ROC-AUC.
    """
    actual = []
    pred = []
    probs_all = []
    
    model.eval()
    pbar = tqdm(test_loader, desc=f"Evaluating")
    thresholds = torch.tensor(thresholds, device=device).view(1, -1)
    for (images, labels) in pbar:
        images, labels = images.to(device), labels.to(device)
        with torch.no_grad():
            logits = model(images)

            probs = torch.sigmoid(logits)

            y_pred = (probs > thresholds).int().cpu().numpy()
            y_actual = labels.cpu().numpy()
    
            actual.append(y_actual)
            pred.append(y_pred)
            probs_all.append(probs.cpu().numpy())
    actual = np.vstack(actual)
    pred = np.vstack(pred)
    probs_all = np.vstack(probs_all)
            
    jaccard = jaccard_score(actual, pred, average='samples', zero_division=0)
    per_class_accuracy = (pred == actual).mean(axis=0) * 100
    per_class_accuracy_dict = {
        disease: acc for disease, acc in zip(task.DISEASE_LABELS, per_class_accuracy)
    }
    f1 = f1_score(actual, pred, average='macro', zero_division=0)
    per_class_f1 = f1_score(actual, pred, average=None, zero_division=0)
    per_class_f1_dict = {
        disease: f1 for disease, f1 in zip(task.DISEASE_LABELS, per_class_f1)
    }
    recall = recall_score(actual, pred, average='macro', zero_division=0)
    precision = precision_score(actual, pred, average='macro', zero_division=0)
    roc_auc = roc_auc_score(actual, probs_all, average='macro')
    per_class_auc = roc_auc_score(actual, probs_all, average=None)
    per_class_auc_dict = {
        disease: auc for disease, auc in zip(task.DISEASE_LABELS, per_class_auc)
    }
    aucpr = average_precision_score(actual, probs_all, average='macro')
    per_class_aucpr = average_precision_score(actual, probs_all, average=None)
    per_class_aucpr_dict = {
        disease: aucpr for disease, aucpr in zip(task.DISEASE_LABELS, per_class_aucpr)
    }

    report = classification_report(actual, pred, target_names=task.DISEASE_LABELS)

    print("TYPE:", type(actual))
    
    results = {
        'Jaccard Similarity': jaccard,
        'per_class_acc': per_class_accuracy_dict,
        'F1_score': f1, 
        'per_class_f1': per_class_f1_dict,
        'Recall': recall,
        'Precision': precision,
        'roc_auc': roc_auc,
        'per_class_auc': per_class_auc_dict,
        'aucpr': aucpr,
        'per_class_aucpr': per_class_aucpr_dict,
        'actual': actual,
        'probs': probs_all,
        'classification_report': report
    }
    return results

In [ ]:
eval_dataset = task.XrayDataset(df_eval_combined, transform=preprocess_transform)
eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

results = evaluate(densenet121, eval_loader, device);

In [ ]:
print(results['classification_report'])

In [ ]:
per_class_auc = results['per_class_auc']

for key, val in per_class_auc.items():
    print(f'{key:20s} {val:4f}')

In [ ]:
per_class_acc = results['per_class_acc']

for key, val in per_class_acc.items():
    print(f'{key:20s} {val:4f}%')

In [ ]:
per_class_f1 = results['per_class_f1']

for key, val in per_class_f1.items():
    print(f'{key:20s} {val:4f}')

In [ ]:
from sklearn.metrics import jaccard_score, f1_score, recall_score, precision_score, roc_auc_score, roc_curve, average_precision_score, RocCurveDisplay

In [ ]:
type(results['actual'])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for i, disease in enumerate(task.DISEASE_LABELS):
    RocCurveDisplay.from_predictions(
        results['actual'][:, i],
        results['probs'][:, i],
        name=disease,
        ax=ax
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="black")
ax.set_title("ROC Curves (Per Class)")
plt.show()

In [ ]:
results['Precision']

In [ ]:
results['Recall']